In [ ]:
MODELS = ["Qwen/Qwen3.5-4B", "google/gemma-3-4b-it", "meta-llama/Llama-3.1-8B-Instruct"]
INPUT_FILE = "/kaggle/input/<dataset>/dev.jsonl"
MAX_NEW_TOKENS = {"classify": 256, "roles": 768}
DRY_RUN = False
OUTPUT_DIR = "/kaggle/working/llm_outputs"


In [ ]:
import importlib.metadata
import subprocess
import sys

VERSIONS = {"python": sys.version.split()[0]}
if not DRY_RUN:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-U",
                           "transformers", "accelerate", "bitsandbytes"])
    import torch
    from huggingface_hub import login
    from kaggle_secrets import UserSecretsClient
    # Authorise this notebook to access HF_TOKEN in Kaggle Add-ons > Secrets.
    token = UserSecretsClient().get_secret("HF_TOKEN")
    login(token=token, add_to_git_credential=False)
    del token
    for package in ("transformers", "accelerate", "bitsandbytes", "torch"):
        VERSIONS[package] = importlib.metadata.version(package)
    if not torch.cuda.is_available():
        raise RuntimeError("Enable a T4 GPU before running")
print(VERSIONS)


In [ ]:
import gc
import hashlib
import json
import re
import time
from datetime import datetime, timezone
from pathlib import Path

PROMPT_VERSIONS = {"classify": "sp1_classify_v2 1.0.0", "roles": "sp1_roles_v2 1.0.0"}

def clean_reply(text):
    cleaned, count = re.subn(r"^\s*<think>.*?</think>\s*", "", text, count=1, flags=re.S)
    return cleaned, count

def input_digest(row):
    prompts = {kind: row[kind + "_prompt"] for kind in PROMPT_VERSIONS}
    return hashlib.sha256(json.dumps(prompts, sort_keys=True, ensure_ascii=False).encode()).hexdigest()

def read_completed(path, model_id, expected):
    """Discard only a partial final write. Reject stale prompts and duplicate records."""
    completed = set()
    path = Path(path)
    if not path.exists():
        return completed
    payload = path.read_bytes()
    if payload and not payload.endswith(b"\n"):
        # Each append is newline terminated. Recover an interrupted final append.
        payload = payload[:payload.rfind(b"\n") + 1]
        path.write_bytes(payload)
    for line in payload.decode("utf-8").splitlines():
        row = json.loads(line)
        pid = row["post_id"]
        if (row["model_id"] != model_id or row["prompt_versions"] != PROMPT_VERSIONS
                or pid not in expected or row.get("input_sha256") != expected[pid]):
            raise ValueError("Resume file belongs to another model, input or prompt version")
        if pid in completed:
            raise ValueError("Duplicate post in resume file")
        if set(row.get("replies", {})) != set(PROMPT_VERSIONS):
            raise ValueError("Incomplete replies in resume file")
        completed.add(pid)
    return completed

def fake_reply(kind):
    # Fixed fake output tests execution, not model accuracy. Roles intentionally have no IDs.
    if kind == "classify":
        return json.dumps({"post_type": "insufficient", "post_type_evidence": "L1",
                           "activity_type": "other", "domain": None,
                           "mode": None, "mode_evidence": None})
    return json.dumps({"title": "none", "multi_session": None, "dates": {}, "times": {}, "urls": {}})

def model_slug(model_id):
    return model_id.replace("/", "__")

def run_record(row, model_id, generate):
    replies, elapsed, tokens, stripped = {}, {}, {}, {}
    for kind in PROMPT_VERSIONS:
        reply, stats = generate(row[kind + "_prompt"], kind)
        replies[kind], stripped[kind] = clean_reply(reply)
        elapsed[kind] = stats["seconds"]
        tokens[kind] = {"input": stats["input_tokens"], "output": stats["output_tokens"]}
    return {"post_id": row["post_id"], "model_id": model_id, "prompt_versions": PROMPT_VERSIONS,
            "input_sha256": input_digest(row), "replies": replies, "seconds": elapsed, "tokens": tokens,
            "peak_vram_gb": stats["peak_vram_gb"], "thinking_stripped": stripped}


In [ ]:
def load_model(model_id):
    from transformers import (AutoConfig, AutoTokenizer, AutoModelForCausalLM,
                              AutoModelForImageTextToText, AutoProcessor, BitsAndBytesConfig)
    config = AutoConfig.from_pretrained(model_id)
    quant = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_quant_type="nf4",
                              bnb_4bit_compute_dtype=torch.float16)
    kwargs = {"quantization_config": quant, "device_map": "auto", "dtype": torch.float16}
    processor = None
    try:
        model = AutoModelForCausalLM.from_pretrained(model_id, **kwargs)
        tokenizer = AutoTokenizer.from_pretrained(model_id)
    except ValueError:
        # Unsupported multimodal config is expected here. Do not swallow auth/OOM errors.
        if not (hasattr(config, "vision_config") and hasattr(config, "text_config")):
            raise
        model = AutoModelForImageTextToText.from_pretrained(model_id, **kwargs)
        processor = AutoProcessor.from_pretrained(model_id)
        tokenizer = processor.tokenizer
    model.eval()
    # Greedy decoding. Remove sampling defaults inherited from generation_config.
    model.generation_config.temperature = None
    model.generation_config.top_p = None
    model.generation_config.top_k = None
    print(model_id, type(model).__name__)
    return model, tokenizer, processor

def make_generate(model, tokenizer, processor, model_id):
    def generate(prompt, kind):
        messages = [{"role": "user", "content": prompt if processor is None else
                     [{"type": "text", "text": prompt}]}]
        template = processor if processor is not None else tokenizer
        options = {"enable_thinking": False} if "qwen" in model_id.lower() else {}
        if processor is not None:
            inputs = template.apply_chat_template(messages, add_generation_prompt=True,
                        tokenize=True, return_dict=True, return_tensors="pt", **options)
        else:
            text = template.apply_chat_template(messages, add_generation_prompt=True,
                                                 tokenize=False, **options)
            inputs = tokenizer(text, return_tensors="pt", add_special_tokens=False)
        inputs = inputs.to(model.device)
        prompt_tokens = inputs["input_ids"].shape[-1]
        context = getattr(model.config, "max_position_embeddings",
                          getattr(getattr(model.config, "text_config", None),
                                  "max_position_embeddings", None))
        if context and prompt_tokens + MAX_NEW_TOKENS[kind] > context:
            raise ValueError("Prompt exceeds model context. Never silently truncate candidate IDs.")
        torch.cuda.synchronize()
        started = time.perf_counter()
        with torch.inference_mode():
            output = model.generate(**inputs, do_sample=False, max_new_tokens=MAX_NEW_TOKENS[kind],
                                    pad_token_id=tokenizer.pad_token_id or tokenizer.eos_token_id)
        torch.cuda.synchronize()
        duration = time.perf_counter() - started
        generated = output[0, prompt_tokens:]
        raw = tokenizer.decode(generated, skip_special_tokens=True)
        peak = max(torch.cuda.max_memory_allocated(i) for i in range(torch.cuda.device_count())) / 2**30
        return raw, {"seconds": duration, "input_tokens": prompt_tokens,
                     "output_tokens": len(generated), "peak_vram_gb": peak}
    return generate


In [ ]:
rows = [json.loads(line) for line in Path(INPUT_FILE).read_text(encoding="utf-8").splitlines() if line.strip()]
if len({row["post_id"] for row in rows}) != len(rows):
    raise ValueError("Duplicate input post IDs")
if any(row.get("prompt_versions") != PROMPT_VERSIONS for row in rows):
    raise ValueError("Input prompt versions do not match")
expected = {row["post_id"]: input_digest(row) for row in rows}
outdir = Path(OUTPUT_DIR)
# Fake outputs never pollute real output/resume files.
if DRY_RUN:
    outdir = outdir / "dry_run"
outdir.mkdir(parents=True, exist_ok=True)
info_path = outdir / "run_info.json"
info = json.loads(info_path.read_text()) if info_path.exists() else {"runs": []}
info["runs"].append({"versions": VERSIONS, "gpu": "CPU fake" if DRY_RUN else
                    [torch.cuda.get_device_name(i) for i in range(torch.cuda.device_count())],
                    "date": datetime.now(timezone.utc).isoformat(), "models": MODELS,
                    "dry_run": DRY_RUN, "input_file": INPUT_FILE,
                    "max_new_tokens": MAX_NEW_TOKENS})
info_path.write_text(json.dumps(info, indent=2) + "\n", encoding="utf-8")

for requested_model in MODELS:
    model_id = "FAKE/" + requested_model if DRY_RUN else requested_model
    path = outdir / (model_slug(model_id) + ".jsonl")
    completed = read_completed(path, model_id, expected)
    pending = [row for row in rows if row["post_id"] not in completed]
    if not pending:
        print(model_id, "already complete", len(completed))
        continue
    model = tokenizer = processor = generate = None
    try:
        if DRY_RUN:
            def generate(prompt, kind):
                return fake_reply(kind), {"seconds": 0.0, "input_tokens": 0,
                                          "output_tokens": 0, "peak_vram_gb": 0.0}
        else:
            model, tokenizer, processor = load_model(model_id)
            generate = make_generate(model, tokenizer, processor, model_id)
        with path.open("a", encoding="utf-8", newline="\n") as stream:
            for index, row in enumerate(pending, 1):
                if not DRY_RUN:
                    for device in range(torch.cuda.device_count()):
                        torch.cuda.reset_peak_memory_stats(device)
                record = run_record(row, model_id, generate)
                stream.write(json.dumps(record, ensure_ascii=False) + "\n")
                stream.flush()
                print(model_id, index, "/", len(pending), "posts")
    finally:
        # Drop the closure before the model: it also holds model/tokenizer references.
        del generate, model, tokenizer, processor
        gc.collect()
        if not DRY_RUN:
            torch.cuda.empty_cache()
print("Outputs:", outdir)
